In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
CACHE_DIR = BASE_DIR / "LSTM_cache_TL/Transformer_exp"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for LSTM. 

In [ ]:
ALL_REGIONS = [
    'CH',
    'NOR',
    'ISL',
    'FR',
]
TARGET_REGIONS = ['SJM']

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
}

run_flag_by_code = {
    'CH': False,
    'NOR': False,
    'ISL': False,
    'FR': False,
    'SJM': False,
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=ALL_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

res_xreg_by_source = {
    region: monthly_cache[region]
    for region in ALL_REGIONS + TARGET_REGIONS
}

### Finetuning glaciers:

#### Automatic picking:

In [ ]:
CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']

# remap cache keys to what the function expects
res_xreg_remapped = {
    region: {
        "df_train": res_xreg_by_source[region]['data_monthly'],
        "df_test": res_xreg_by_source[region]['data_monthly_aug'],
    }
    for region in ALL_REGIONS
}

scaler_m, scaler_s, scaler_all = build_global_scalers_multi_source_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(f"blur_m={blur_m:.4f}  blur_s={blur_s:.4f}  blur_joint={blur_joint:.4f}")

In [ ]:
def rank_glaciers_by_distance_to_target(
    res_xreg_by_source: dict,
    training_regions: list,
    test_region: str,
    monthly_cols: list,
    static_cols: list,
    scaler_m,
    scaler_s,
    blur_joint: float,
    glacier_col: str = "GLACIER",
    id_col: str = "ID",
    elev_diff_col: str = "ELEVATION_DIFFERENCE",
    max_samples: int = 2000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    For each glacier in training_regions, compute its Sinkhorn distance
    to the full test region distribution.
    Uses same feature construction as split_pool_holdout_sinkhorn_v2 (joint mode).
    Returns a DataFrame sorted by distance ascending (closest to test first).
    """

    # --- feature matrix builder (mirrors split_pool_holdout_sinkhorn_v2) ---
    def _build_X(df):
        pure_static = [c for c in static_cols if c != elev_diff_col]

        def _stake_topo(df_):
            parts = [df_.groupby(id_col)[pure_static].first()]
            if elev_diff_col in static_cols:
                parts.append(df_.groupby(id_col)[[elev_diff_col]].mean())
            return pd.concat(parts,
                             axis=1)[static_cols].to_numpy(dtype=np.float64)

        id_codes_cat = pd.Categorical(df[id_col]).codes
        Xs_id = scaler_s.transform(_stake_topo(df))
        topo_per_row = Xs_id[id_codes_cat]
        Xm_all = scaler_m.transform(
            df[monthly_cols].to_numpy(dtype=np.float64))
        return np.hstack([Xm_all, topo_per_row]).astype(np.float32)

    loss_fn = SamplesLoss(
        loss="sinkhorn",
        p=2,
        blur=blur_joint,
        scaling=0.9,
        debias=True,
        backend="tensorized",
    )

    def _sinkhorn(Xa, Xb):
        if len(Xa) < 2 or len(Xb) < 2:
            return float("nan")
        if len(Xa) > max_samples:
            Xa = Xa[np.random.default_rng(seed).choice(len(Xa),
                                                       max_samples,
                                                       replace=False)]
        if len(Xb) > max_samples:
            Xb = Xb[np.random.default_rng(seed).choice(len(Xb),
                                                       max_samples,
                                                       replace=False)]
        ta = torch.as_tensor(Xa, dtype=torch.float32)
        tb = torch.as_tensor(Xb, dtype=torch.float32)
        wa = torch.ones(len(ta)) / len(ta)
        wb = torch.ones(len(tb)) / len(tb)
        with torch.no_grad():
            return float(loss_fn(wa, ta, wb, tb).item())

    # --- build test region feature matrix once ---
    df_test = res_xreg_by_source[test_region]['data_monthly']
    X_test = _build_X(df_test)
    print(f"Test region ({test_region}): {len(X_test)} rows, "
          f"{df_test[glacier_col].nunique()} glaciers")

    # --- per-glacier distances ---
    rows = []
    for region in training_regions:
        df_region = res_xreg_by_source[region]['data_monthly']
        glaciers = df_region[glacier_col].unique()
        print(f"\n{region}: {len(glaciers)} glaciers")

        for glacier in glaciers:
            df_gl = df_region[df_region[glacier_col] == glacier]
            if len(df_gl) < 2:
                print(f"  skipping {glacier} (< 2 rows)")
                continue

            X_gl = _build_X(df_gl)
            d = _sinkhorn(X_gl, X_test)

            rows.append({
                'glacier': glacier,
                'region': region,
                'distance': d,
                'n_meas': len(df_gl),
            })
            print(f"  {glacier:40s}  d={d:.4f}  n={len(df_gl)}")

    df_ranked = (pd.DataFrame(rows).sort_values('distance').reset_index(
        drop=True))
    df_ranked['rank'] = np.arange(len(df_ranked))
    df_ranked['cum_meas'] = df_ranked['n_meas'].cumsum()
    df_ranked['cum_frac'] = df_ranked['cum_meas'] / df_ranked['n_meas'].sum()

    print(f"\n=== Ranking complete ===")
    print(f"  Total glaciers : {len(df_ranked)}")
    print(f"  Total meas     : {df_ranked['n_meas'].sum()}")
    print(f"  Distance range : [{df_ranked['distance'].min():.4f} "
          f"— {df_ranked['distance'].max():.4f}]")
    print(f"\n  Closest 10 to {test_region}:")
    print(
        df_ranked.head(10)[[
            'rank', 'glacier', 'region', 'distance', 'n_meas', 'cum_frac'
        ]].to_string(index=False))
    print(f"\n  Furthest 10 from {test_region}:")
    print(
        df_ranked.tail(10)[[
            'rank', 'glacier', 'region', 'distance', 'n_meas', 'cum_frac'
        ]].to_string(index=False))

    return df_ranked

#### Split glaciers of training sets:

In [ ]:
df_ranked_sjm = rank_glaciers_by_distance_to_target(
    res_xreg_by_source=res_xreg_by_source,
    training_regions=ALL_REGIONS,  # ['CH', 'NOR', 'ISL', 'FR']
    test_region='SJM',
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    blur_joint=blur_joint,
    seed=cfg.seed,
)

# save so you don't have to rerun
df_ranked_sjm.to_csv(BASE_DIR / "glacier_ranking_sjm.csv", index=False)

## Train transformers:

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))

# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[0]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

In [ ]:
months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']

In [ ]:
FRACS = [0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.0]
N_RANDOM_SEEDS = 10


def build_glacier_subsets(df_ranked, fracs, n_random_seeds, seed=42):
    """
    For each fraction build:
      - closest_first: top-k glaciers by distance
      - furthest_first: bottom-k glaciers by distance  
      - random: N seeds
    All matched to same cumulative measurement count.
    """
    total_meas = df_ranked['n_meas'].sum()
    subsets = {}

    for frac in fracs:
        pct = int(frac * 100)
        target_meas = int(round(frac * total_meas))
        subsets[pct] = {}

        # --- closest first ---
        cumsum, glaciers_close = 0, []
        for _, row in df_ranked.iterrows():
            glaciers_close.append(row['glacier'])
            cumsum += row['n_meas']
            if cumsum >= target_meas:
                break
        subsets[pct]['closest'] = glaciers_close

        # --- furthest first ---
        cumsum, glaciers_far = 0, []
        for _, row in df_ranked.iloc[::-1].iterrows():
            glaciers_far.append(row['glacier'])
            cumsum += row['n_meas']
            if cumsum >= target_meas:
                break
        subsets[pct]['furthest'] = glaciers_far

        # --- random ---
        rng = np.random.default_rng(seed)
        subsets[pct]['random'] = []
        all_glaciers = df_ranked['glacier'].tolist()
        for seed_idx in range(n_random_seeds):
            s = int(rng.integers(0, 2**31))
            rng_local = np.random.default_rng(s)
            shuffled = list(rng_local.permutation(all_glaciers))
            cumsum, glaciers_rnd = 0, []
            for g in shuffled:
                glaciers_rnd.append(g)
                cumsum += int(df_ranked.loc[df_ranked['glacier'] == g,
                                            'n_meas'].iloc[0])
                if cumsum >= target_meas:
                    break
            subsets[pct]['random'].append({
                'seed_idx': seed_idx,
                'seed': s,
                'glaciers': glaciers_rnd,
                'n_meas': cumsum,
            })

        n_close = len(glaciers_close)
        n_far = len(glaciers_far)
        n_rnd = np.mean([len(s['glaciers']) for s in subsets[pct]['random']])
        print(f"  {pct:3d}%  target_meas={target_meas}  "
              f"close={n_close}gl  far={n_far}gl  rnd~{n_rnd:.1f}gl")

    return subsets

In [ ]:
# --- 1. Build glacier subsets at each fraction ---
gl_subsets_sjm = build_glacier_subsets(
    df_ranked=df_ranked_sjm,
    fracs=FRACS,
    n_random_seeds=N_RANDOM_SEEDS,
    seed=cfg.seed,
)

# --- 2. Build pooled assets per subset ---
# dummy splits since SJM is our test — no within-region holdout needed
dummy_splits = {
    region: {
        "holdout_glaciers": [],
        "pool_glaciers": []
    }
    for region in ALL_REGIONS
}
gl_subsets_sjm

In [ ]:
def build_assets_from_glacier_list(
    glaciers: list,
    df_ranked: pd.DataFrame,
    res_xreg_by_source: dict,
    monthly_cols: list,
    static_cols: list,
    cfg,
    months_head_pad,
    months_tail_pad,
    cache_path=None,
    force_recompute=False,
):
    """
    Build MBSequenceDataset from a specific list of glaciers
    (drawn from multiple regions).
    """
    import joblib

    if cache_path and not force_recompute and os.path.exists(cache_path):
        print(f"Loading from cache: {cache_path}")
        cached = joblib.load(cache_path)
        return cached["assets"]

    # map glacier -> region
    gl_to_region = df_ranked.set_index('glacier')['region'].to_dict()

    # collect df_loss and df_full per region
    train_dfs_loss, train_dfs_full = [], []
    for region in ALL_REGIONS:
        region_glaciers = [
            g for g in glaciers if gl_to_region.get(g) == region
        ]
        if not region_glaciers:
            continue

        df_loss = res_xreg_by_source[region]['data_monthly']
        df_full = res_xreg_by_source[region]['data_monthly_aug']

        train_dfs_loss.append(
            df_loss[df_loss['GLACIER'].isin(region_glaciers)])
        train_dfs_full.append(
            df_full[df_full['GLACIER'].isin(region_glaciers)])

    df_pooled_loss = pd.concat(train_dfs_loss, ignore_index=True)
    df_pooled_full = pd.concat(train_dfs_full, ignore_index=True)

    mbm.utils.seed_all(cfg.seed)

    ds = build_combined_LSTM_dataset(
        df_loss=df_pooled_loss,
        df_full=df_pooled_full,
        monthly_cols=monthly_cols,
        static_cols=static_cols,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        normalize_target=True,
        expect_target=True,
        show_progress=False,
    )

    train_idx, val_idx = mbm.data_processing.MBSequenceDataset.split_indices(
        len(ds), val_ratio=0.2, seed=cfg.seed)

    assets = {"ds_train": ds, "train_idx": train_idx, "val_idx": val_idx}

    if cache_path:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        joblib.dump({"assets": assets}, cache_path, compress=3)
        print(f"Saved to cache: {cache_path}")

    return assets


# SJM test dataset (full, zero-shot)
ds_sjm_test = build_combined_LSTM_dataset(
    df_loss=res_xreg_by_source['SJM']['data_monthly'],
    df_full=res_xreg_by_source['SJM']['data_monthly_aug'],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['SJM']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['SJM']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)
print(f"SJM test: {len(ds_sjm_test)} sequences")

In [ ]:
def train_or_load_one_source_model(
        cfg,
        key: str,
        lstm_assets: dict,
        best_params: dict,
        device,
        models_dir="models",
        prefix="lstm_xreg",
        train_flag=True,
        force_retrain=True,
        epochs=150,
        batch_size_train=64,
        batch_size_val=128,
        verbose=True,
        date=None,
        model_type="lstm",  # "lstm" or "transformer"
):
    """Train or load a single source-region model. No test set needed."""
    assert model_type in ("lstm", "transformer"), \
        f"model_type must be 'lstm' or 'transformer', got '{model_type}'"

    # ---- resolve model class once ----
    model_cls = mbm.models.LSTM_MB if model_type == "lstm" else mbm.models.Transformer_MB

    run_date = datetime.now().strftime("%Y-%m-%d") if date is None else date
    os.makedirs(models_dir, exist_ok=True)
    model_filename = os.path.join(models_dir, f"{prefix}_{key}_{run_date}.pt")

    # ---- these are the only two lines that used to be hardcoded ----
    model = model_cls.build_model_from_params(cfg,
                                              best_params,
                                              device,
                                              verbose=verbose)
    loss_fn = model_cls.resolve_loss_fn(best_params)

    # everything below is unchanged
    if train_flag and (not force_retrain) and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    if not train_flag and not os.path.exists(model_filename):
        raise FileNotFoundError(
            f"No checkpoint found for {key}: {model_filename}")

    if not train_flag and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    # --- Train ---
    mbm.utils.seed_all(cfg.seed)

    ds_train = lstm_assets["ds_train"]
    train_idx = lstm_assets["train_idx"]
    val_idx = lstm_assets["val_idx"]

    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_train)

    train_dl, val_dl = ds_train_copy.make_loaders(
        train_idx=train_idx,
        val_idx=val_idx,
        batch_size_train=batch_size_train,
        batch_size_val=batch_size_val,
        seed=cfg.seed,
        fit_and_transform=True,
        shuffle_train=True,
        use_weighted_sampler=True,
        verbose=verbose,
    )

    if os.path.exists(model_filename):
        os.remove(model_filename)
        if verbose:
            print(f"Deleted existing model file: {model_filename}")

    history, best_val, best_state = model.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        clip_val=1,
        sched_factor=0.5,
        sched_patience=6,
        sched_threshold=0.01,
        sched_threshold_mode="rel",
        sched_cooldown=1,
        sched_min_lr=1e-6,
        es_patience=15,
        es_min_delta=1e-4,
        log_every=5,
        verbose=verbose,
        save_best_path=model_filename,
        loss_fn=loss_fn,
    )

    if verbose:
        plot_history_lstm(history)

    state = torch.load(model_filename, map_location=device)
    model.load_state_dict(state)

    return model, model_filename, {"history": history, "best_val": best_val}

In [ ]:
# --- 3. Train transformer for each subset ---
TRAIN_FLAG = True
MODEL_DATE = "2026-05-21"
CACHE_DIR_RANK = CACHE_DIR / "ranking_sjm"
os.makedirs(CACHE_DIR_RANK, exist_ok=True)

ranking_results = []

for pct, subset in gl_subsets_sjm.items():
    print(f"\n{'='*60}")
    print(f"  Fraction: {pct}%")

    strategies = {
        "closest": subset['closest'],
        "furthest": subset['furthest'],
        **{
            f"random_{s['seed_idx']}": s['glaciers']
            for s in subset['random']
        },
    }

    for strategy_name, glaciers in strategies.items():
        print(f"\n  --- {strategy_name} ({len(glaciers)} glaciers) ---")

        cache_path = str(CACHE_DIR_RANK /
                         f"assets_{pct}pct_{strategy_name}.joblib")

        assets = build_assets_from_glacier_list(
            glaciers=glaciers,
            df_ranked=df_ranked_sjm,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=cache_path,
            force_recompute=False,
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
        )

        model, model_path, info = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{pct}pct_{strategy_name}",
            lstm_assets=assets,
            best_params=best_params_gs,
            device=device,
            models_dir=BASE_DIR / "models/ranking_sjm",
            prefix="transformer_rank",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=150,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )

        # --- evaluate on SJM ---
        ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            assets["ds_train"])
        ds_scaler.make_loaders(
            train_idx=assets["train_idx"],
            val_idx=assets["val_idx"],
            fit_and_transform=True,
            seed=cfg.seed,
            verbose=False,
        )
        ds_sjm_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds_sjm_test)
        sjm_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_sjm_copy,
            ds_scaler,
            batch_size=128,
            seed=cfg.seed,
        )
        metrics, _ = model.evaluate_with_preds(device, sjm_dl, ds_sjm_copy)

        is_random = strategy_name.startswith("random")
        print(f"    RMSE_a={metrics['RMSE_annual']:.3f}  "
              f"RMSE_w={metrics['RMSE_winter']:.3f}  "
              f"R2_a={metrics['R2_annual']:.3f}")

        ranking_results.append({
            "pct":
            pct,
            "strategy":
            "random" if is_random else strategy_name,
            "seed_idx":
            int(strategy_name.split("_")[1]) if is_random else None,
            "n_glaciers":
            len(glaciers),
            "n_sequences":
            len(assets["ds_train"]),
            **{
                k: round(v, 3)
                for k, v in metrics.items()
            },
        })

# --- save results ---
df_ranking_results = pd.DataFrame(ranking_results)
df_ranking_results.to_csv(BASE_DIR / "ranking_sjm_results.csv", index=False)
print(
    df_ranking_results.groupby(["pct",
                                "strategy"]).mean(numeric_only=True).round(3))

In [ ]:
import matplotlib.ticker as mtick

METRICS = ["RMSE_annual", "RMSE_winter", "R2_annual", "R2_winter"]
PCTS_PLOT = sorted(df_ranking_results["pct"].unique())

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes = axes.flatten()

for ax, metric in zip(axes, METRICS):
    is_r2 = metric.startswith("R2")

    # random: mean ± std + scatter
    df_rnd = df_ranking_results[df_ranking_results["strategy"] == "random"]
    agg = df_rnd.groupby("pct")[metric].agg(["mean", "std"]).reindex(PCTS_PLOT)

    ax.plot(PCTS_PLOT,
            agg["mean"],
            marker="o",
            color="steelblue",
            label="Random (mean)",
            zorder=4)
    ax.fill_between(PCTS_PLOT,
                    agg["mean"] - agg["std"],
                    agg["mean"] + agg["std"],
                    alpha=0.2,
                    color="steelblue",
                    label="Random ±1 std")
    for _, row in df_rnd.iterrows():
        ax.scatter(row["pct"],
                   row[metric],
                   color="steelblue",
                   alpha=0.25,
                   s=15,
                   zorder=3)

    mean_rnd = agg["mean"].values
    x = np.array(PCTS_PLOT)

    # closest
    df_close = df_ranking_results[df_ranking_results["strategy"] == "closest"]
    close_vals = df_close.set_index("pct")[metric].reindex(PCTS_PLOT).values
    ax.plot(PCTS_PLOT,
            close_vals,
            marker="D",
            linestyle="--",
            color="darkorange",
            label="Closest (Sinkhorn)",
            zorder=5,
            markersize=8)
    better = close_vals < mean_rnd if not is_r2 else close_vals > mean_rnd
    ax.fill_between(x,
                    mean_rnd,
                    close_vals,
                    where=better,
                    alpha=0.12,
                    color="green")
    ax.fill_between(x,
                    mean_rnd,
                    close_vals,
                    where=~better,
                    alpha=0.12,
                    color="red")

    # furthest
    df_far = df_ranking_results[df_ranking_results["strategy"] == "furthest"]
    far_vals = df_far.set_index("pct")[metric].reindex(PCTS_PLOT).values
    ax.plot(PCTS_PLOT,
            far_vals,
            marker="s",
            linestyle="-.",
            color="#b2182b",
            label="Furthest (Sinkhorn)",
            zorder=5,
            markersize=8)
    better_f = far_vals < mean_rnd if not is_r2 else far_vals > mean_rnd
    ax.fill_between(x,
                    mean_rnd,
                    far_vals,
                    where=better_f,
                    alpha=0.08,
                    color="green",
                    hatch="//",
                    linewidth=0)
    ax.fill_between(x,
                    mean_rnd,
                    far_vals,
                    where=~better_f,
                    alpha=0.08,
                    color="red",
                    hatch="//",
                    linewidth=0)

    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric, fontsize=10)
    ax.set_xlabel("Training data fraction (%)", fontsize=10)
    ax.set_xticks(PCTS_PLOT)
    ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("%d%%"))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, framealpha=0.9)
    apply_nature_style(ax, fontsize=NATURE_SPECS["font_min_pt"], box=True)

fig.suptitle(
    f"Transformer: closest vs furthest vs random glaciers → SJM (zero-shot)\n"
    f"Training pool: {' + '.join(ALL_REGIONS)}  |  {N_RANDOM_SEEDS} random seeds",
    fontsize=13,
)
fig.tight_layout()
plt.savefig(BASE_DIR / "ranking_sjm_transformer.pdf", bbox_inches="tight")
plt.savefig(BASE_DIR / "ranking_sjm_transformer.png",
            dpi=150,
            bbox_inches="tight")
plt.show()